# 06 · CUB70 and MCBM — does minimality change visibility grounding?

MCBM pulls each internal value `z_j` toward the label-only target `6c_j−3`. This restricts information stored in `z_j`; it does not specify which pixels compute it.

Using the CUB70 masks, we ask whether originally-positive concepts remain high when their named part is not visible, and whether that relationship changes with minimality weight `γ`.

These are natural photographs, not rendered deletions. The valid quantity is `c_pred_j` or `z_j` split by mask visibility—not `p_removed/p_intact`.


In [ ]:
import os,sys,re
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
CURATED=Path(os.environ["CURATED_DATA"]); CWD=Path.cwd();REPO=CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0,str(REPO/"analysis")); sys.path.insert(0,str(REPO/"data"/"cub70"))
from occlusion import (attach_visibility,z_by_visibility,grounding_violation_rate,
    quartile_grounding,within_species_visibility_effect)
from relabel_cub_with_cub70 import coarse_visibility
try:
    from plotting import set_paper_style,PALETTE; set_paper_style(); MCBM_C=PALETTE["MCBM"]
except Exception: MCBM_C="#D55E00"
def need(p,how):
    p=Path(p)
    if not p.exists(): print(f"[pending] {p}\n  produce it: {how}")
    return p.exists()
vis_path=CURATED/"cub70_visibility.parquet"; VIS=None
if need(vis_path,"bash data/cub70/prepare_all.sh"):
    VIS=coarse_visibility(pd.read_parquet(vis_path),.001).rename(columns={"coarse":"part"})


## 1 · Full-CUB MCBM on masked CUB70 images

Start with one full-CUB MCBM setting. Among rows with original `c_j=1`, compare

`c_pred_j | visible=1` and `c_pred_j | visible=0`,

and calculate

`violation_rate = P(c_pred_j≥0.5 | c_j=1, visible=0)`.

High violation means the named part is not necessary for the positive output on these photographs. It does not identify species as the unique alternative source.


In [ ]:
SEED=1; GAMMA=1; tag=str(GAMMA).replace(".","p")
full_path=CURATED/"cub70_eval"/f"cub-mcbm-g{tag}-s{SEED}.parquet"; J_FULL=None
if VIS is not None and need(full_path,"CONFIGS='cub-mcbm-g1' bash analysis/cub70_prepare_analysis.sh"):
    E=pd.read_parquet(full_path); J_FULL=attach_visibility(E[E.part!=""],VIS)
    Z=z_by_visibility(J_FULL); V=grounding_violation_rate(J_FULL)
    display(Z.round(3)); display(V.round(3))
    P=Z.pivot(index="part",columns="visible",values="prob_mean").rename(columns={False:"occluded",True:"visible"})
    fig,ax=plt.subplots(figsize=(7,3.5)); x=np.arange(len(P));w=.38
    ax.bar(x-w/2,P.get("occluded"),w,label="mask absent",color="#CC79A7")
    ax.bar(x+w/2,P.get("visible"),w,label="mask visible",color=MCBM_C)
    ax.set_xticks(x);ax.set_xticklabels(P.index,rotation=30,ha="right");ax.set_ylim(0,1)
    ax.set_ylabel("mean c_pred_j for original c_j=1");ax.set_title(f"Full-CUB MCBM γ={GAMMA}: output versus visibility");ax.legend();plt.show()


## 2 · Visibility area, not merely visible/absent

Within each part, divide mask `area_frac` into quartiles and compute `mean(c_pred_j)` for original `c_j=1` rows. A rising curve shows sensitivity to available part evidence; a flat high curve shows that the answer survives across visibility levels.


In [ ]:
if J_FULL is not None:
    Q=quartile_grounding(J_FULL);display(Q.round(4))
    fig,ax=plt.subplots(figsize=(7,3.6))
    for part,d in Q.groupby("part"):ax.plot(d.qbin,d.prob_mean,"o-",label=part)
    ax.set_xlabel("within-part mask-area quartile");ax.set_ylabel("mean c_pred_j for original c_j=1")
    ax.set_ylim(0,1);ax.set_title(f"Full-CUB MCBM γ={GAMMA}: visibility dose response");ax.legend(ncol=2);plt.show()


## 3 · Evaluation-label disagreement

For test images only, define `c_j^vis=c_j×visible_part(j)` and count positive labels changed to zero. This quantifies what the masks reveal that species-level labels omit.

Because CUB70 has no training masks, this is not a relabeled MCBM training comparison. The same diagnostic is shared with notebook 05 and should not be interpreted as an MCBM effect.


In [ ]:
diag_path=CURATED/"cub70_relabel_diagnostics.parquet"
if need(diag_path,"python data/cub70/relabel_cub_with_cub70.py"):
    D=pd.read_parquet(diag_path);pos=D[D.original_label==1]
    R=pos.groupby("part").agg(original_positive=("flipped","size"),flipped=("flipped","sum"))
    R["flip_rate_given_positive"]=R.flipped/R.original_positive;display(R.round(3))
    fig,ax=plt.subplots(figsize=(7,3.3));ax.bar(R.index,R.flip_rate_given_positive,color=MCBM_C)
    ax.set_ylim(0,1);ax.set_ylabel("P(mask absent | original c_j=1)")
    ax.set_title("CUB70 evaluation labels contradicted by visibility masks");plt.xticks(rotation=30,ha="right");plt.show()


## 4 · Does `γ` change visibility grounding?

For every trained CUB70 MCBM, compute

`violation_rate(γ)=P(c_pred_j≥0.5 | c_j=1, visible=0, γ)`.

If minimality creates pixel grounding, this rate should fall while ordinary concept accuracy remains healthy. If `z_j` compresses but the violation rate remains high, MCBM changed representation content without correcting its pixel source. Every `γ` needs multiple seeds before a trend is claimed.


In [ ]:
if VIS is not None:
    rows=[]
    for path in sorted((CURATED/"cub70_eval").glob("cub70-mcbm-g*-s*.parquet")):
        m=re.match(r"cub70-mcbm-g([0-9p]+)-s(\d+)",path.stem)
        if not m:continue
        gamma=float(m.group(1).replace("p","."));seed=int(m.group(2))
        J=attach_visibility(pd.read_parquet(path).query("part != ''"),VIS)
        V=grounding_violation_rate(J)
        for r in V.itertuples():rows.append(dict(gamma=gamma,seed=seed,part=r.part,n=r.n_occluded,rate=r.violation_rate))
    if rows:
        T=pd.DataFrame(rows);display(T.round(3))
        G=T.groupby(["gamma","part"]).rate.agg(["mean","std","count"]).reset_index()
        fig,ax=plt.subplots(figsize=(8,4))
        for part,d in G.groupby("part"):
            ax.errorbar(d.gamma.replace(0,.03),d["mean"],yerr=d["std"].fillna(0),marker="o",label=part)
        ax.set_xscale("log");ax.set_ylim(0,1);ax.set_xlabel("γ (0 shown at 0.03)")
        ax.set_ylabel("P(c_pred_j≥0.5 | c_j=1, mask absent)");ax.set_title("CUB70 MCBM: visibility violation versus γ");ax.legend(ncol=2);plt.show()
    else:print("[pending] export CUB70 MCBM eval tables for the γ sweep")


## 5 · Full-CUB-trained versus CUB70-trained MCBM

Evaluate both models on the same masked test images. This tests whether restricting the species task to 70 classes changes the visibility relationship. It is descriptive rather than a relabeling intervention: both models used original training labels.


In [ ]:
cub70_path=CURATED/"cub70_eval"/f"cub70-mcbm-g{tag}-s{SEED}.parquet"
if VIS is not None and J_FULL is not None and need(cub70_path,"train/export cub70-mcbm at the same γ"):
    J70=attach_visibility(pd.read_parquet(cub70_path).query("part != ''"),VIS)
    rows=[]
    for name,J in [("full-CUB trained",J_FULL),("CUB70 trained",J70)]:
        for r in grounding_violation_rate(J).itertuples():
            rows.append(dict(model=name,part=r.part,n=r.n_occluded,violation_rate=r.violation_rate))
    C=pd.DataFrame(rows);display(C.round(3));H=C.pivot(index="part",columns="model",values="violation_rate")
    H.plot.bar(figsize=(8,3.6),ylim=(0,1),color=[MCBM_C,"#009E73"])
    plt.ylabel("visibility violation rate");plt.title(f"Same masks, MCBM γ={GAMMA}: training partition comparison")
    plt.xticks(rotation=30,ha="right");plt.show()


## 6 · Species-matched minimality test

<!-- MCBM SPECIES-MATCHED CONTROL -->
The raw visibility comparison can mix species. For each `γ` and seed, repeat the comparison inside a fixed `(species y, concept j)` group and retain only groups with both visible and occluded photographs. If minimality improves local grounding, the within-species difference `mean(c_pred_j|visible)−mean(c_pred_j|occluded)` should grow consistently with `γ`.


In [ ]:
if VIS is not None:
    matched_rows=[]
    for path in sorted((CURATED/"cub70_eval").glob("cub70-mcbm-g*-s*.parquet")):
        m=re.match(r"cub70-mcbm-g([0-9p]+)-s(\d+)",path.stem)
        if not m:continue
        gamma=float(m.group(1).replace("p","."));seed=int(m.group(2))
        J=attach_visibility(pd.read_parquet(path).query("part != ''"),VIS)
        for r in within_species_visibility_effect(J).itertuples():
            matched_rows.append(dict(gamma=gamma,seed=seed,part=r.part,
                n_matched_groups=r.n_matched_groups,effect=r.visible_minus_occluded))
    if matched_rows:
        M=pd.DataFrame(matched_rows);display(M.round(4))
        G=M.groupby(["gamma","part"]).effect.agg(["mean","std","count"]).reset_index()
        fig,ax=plt.subplots(figsize=(8,4))
        for part,d in G.groupby("part"):
            ax.errorbar(d.gamma.replace(0,.03),d["mean"],yerr=d["std"].fillna(0),marker="o",label=part)
        ax.set_xscale("log");ax.axhline(0,color="black",lw=1)
        ax.set_xlabel("γ (0 shown at 0.03)");ax.set_ylabel("within-(species, concept) visibility effect")
        ax.set_title("Does minimality increase species-controlled grounding?");ax.legend(ncol=2);plt.show()
    else:print("[pending] export CUB70 MCBM evaluation tables")


## Conclusion boundary

A decrease in raw violation rate is not enough if the species-matched effect does not change. Conversely, a positive matched effect shows sensitivity to visible part evidence but still does not prove that every occluded answer came from species. Multiple seeds are required before interpreting a `γ` trend.
